# 01 — Data Understanding and Initial Profiling

**Project:** AI Usage and Impact on Students and Professionals

## Learning goals
By the end of this notebook, you should be able to:
- Load a CSV file with Pandas.
- inspect its size, columns, and data types.
- distinguish missing values from duplicate records.
- identify categorical and numerical fields.
- produce an initial data-quality profile.

**Important:** This notebook does not clean or overwrite the original data. We are investigating first.

## 1. Import the libraries

Pandas helps us work with tabular data. NumPy provides numerical tools. We will import Matplotlib and Seaborn later when we begin visual analysis.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

## 2. Load the original CSV

The file is stored in `data/raw/`. We preserve this original copy and will create a separate processed file after agreeing on cleaning decisions.

In [ ]:
data_path = "../data/raw/AI_Usage_and_Impact_on_Students_and_Professionals.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")

## 3. Inspect the shape and preview

`shape` returns (number of rows, number of columns). `head()` displays the first five records.

In [ ]:
print("Rows and columns:", df.shape)
display(df.head())

## 4. Inspect column names and data types

A column's data type tells us how Pandas currently interprets its values. A numeric-looking column can still be stored as text if values contain units such as `hrs` or currency symbols.

In [ ]:
df.info()

In [ ]:
display(pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "unique_non_null": df.nunique(dropna=True).values
}))

## 5. Check missing values

A missing value is not automatically an error. We need to understand how much is missing and whether it is concentrated in particular fields or groups.

In [ ]:
missing_profile = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_percent=lambda x: (x["missing_count"] / len(df) * 100).round(2))
      .sort_values("missing_percent", ascending=False)
)

display(missing_profile)

## 6. Check duplicate rows and repeated user IDs

These are different checks:
- An exact duplicate row repeats every field.
- A repeated `User_ID` means the identifier appears more than once; it needs investigation, but it is not automatically safe to delete.

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Repeated User_ID records:", df["User_ID"].duplicated(keep=False).sum())

repeated_ids = df[df["User_ID"].duplicated(keep=False)].sort_values("User_ID")
display(repeated_ids)

## 7. Identify categorical and numeric-looking fields

The source description says the dataset contains demographic, education/employment, AI usage, and impact information. We will compare this description with the actual columns.

In [ ]:
categorical_candidates = [
    "Gender", "Country", "User_Type", "Education_Level",
    "Profession", "AI_Tool", "AI_Purpose", "Would_Recommend"
]

numeric_candidates = [
    "Age", "Monthly_Income", "AI_Usage_Hours_Per_Day",
    "Monthly_AI_Cost", "Productivity_Score", "Accuracy_Rating",
    "Satisfaction_Score", "Time_Saved_Hours_Per_Week",
    "Work_or_Study_Hours_Per_Day"
]

print("Categorical candidates present:")
print([c for c in categorical_candidates if c in df.columns])

print("\nNumeric candidates present:")
print([c for c in numeric_candidates if c in df.columns])

## 8. Inspect category labels

Look for differences such as `Male` versus `male`, or `ChatGPT` versus `Chat GPT`. Do not standardise them until we agree on a mapping.

In [ ]:
for column in categorical_candidates:
    if column in df.columns:
        print(f"\n{column}")
        print(df[column].value_counts(dropna=False).head(25))

## 9. Compare the CSV with the supplied description

The description mentions `AI_Experience_Years`, but the CSV's 19 columns do not currently include that field. Record this as a documentation/data discrepancy; do not invent the missing column.

In [ ]:
description_claimed_column = "AI_Experience_Years"

print("Does the CSV contain AI_Experience_Years?",
      description_claimed_column in df.columns)
print("\nActual columns:")
print(df.columns.tolist())

## 10. Initial questions for the analyst

Answer these in your own words before moving to data cleaning:

1. What does one row appear to represent?
2. Why should we investigate repeated `User_ID` values before deleting records?
3. Why might `Monthly_Income` need a currency clarification?
4. Why is `AI_Experience_Years` a data documentation issue?
5. Why should we preserve the raw CSV?

## Phase 1 checkpoint

Save your observations in `reports/`. Do not modify the raw CSV. The next notebook will focus on validating data quality and agreeing on cleaning rules.